# Lakelink Fiber — Span Risk Classifier v2: Model Monitoring & Drift Detection

**Endpoint:** `span-risk-classifier-v2`  
**Model:** `cdm_tmforum.provable_resilience_intelligence.span_risk_classifier` v2 (XGBoost)  
**Inference Table:** `cdm_tmforum.provable_resilience_intelligence.span_risk_classifier_payload`  
**AI Gateway:** Rate limit 200 req/min  

This notebook monitors:
1. **Feature drift** — PSI (Population Stability Index) per feature vs. training baseline
2. **Prediction distribution** — Risk label proportions and shift detection
3. **Latency SLA** — P50/P95/P99 response times
4. **Data quality** — Null rates, out-of-range features, schema compliance
5. **Alerting** — Threshold-based flags for drift, latency, anomalous risk rates

In [0]:
# ── Configuration ──────────────────────────────────────────────────────
CATALOG = "cdm_tmforum"
SCHEMA = "provable_resilience_intelligence"
ENDPOINT_NAME = "span-risk-classifier-v2"
INFERENCE_TABLE = f"{CATALOG}.{SCHEMA}.span_risk_classifier_payload"
TRAINING_TABLE = f"{CATALOG}.{SCHEMA}.physical_resource_span"
MODEL_NAME = f"{CATALOG}.{SCHEMA}.span_risk_classifier"
MODEL_VERSION = 2

# Thresholds
PSI_WARNING = 0.1    # PSI > 0.1 = moderate drift
PSI_CRITICAL = 0.25  # PSI > 0.25 = significant drift
LATENCY_P95_SLA_MS = 500  # 500ms P95 SLA
RISK_RATE_LOW = 0.02   # Flag if risk rate drops below 2%
RISK_RATE_HIGH = 0.15  # Flag if risk rate exceeds 15%
NULL_THRESHOLD = 0.05  # Flag if >5% nulls in any feature

# 22 model features
FEATURE_COLS = [
    "fiber_count", "wavelength_count", "length_km", "span_age_years",
    "total_alarms", "distinct_alarm_types", "critical_alarm_count", "major_alarm_count",
    "severe_alarm_count", "osnr_degrade_count", "ber_high_count", "fiber_cut_count",
    "los_count", "avg_alarm_duration_hrs", "unack_alarm_count", "days_since_last_critical",
    "srlg_count", "srlg_type_count", "srlg_critical_count", "srlg_severe_count",
    "service_count", "spans_in_conduit"
]

print(f"Monitoring: {ENDPOINT_NAME}")
print(f"Inference table: {INFERENCE_TABLE}")
print(f"Feature count: {len(FEATURE_COLS)}")

In [0]:
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

# ── Training baseline: compute feature distributions from the training data ──
# We compute percentile bins and summary stats from the actual training features
# by joining physical_resource_span with alarm/SRLG aggregates (same as training pipeline)

baseline_query = f"""
WITH span_alarm_features AS (
  SELECT
    s.span_id,
    s.fiber_count, s.wavelength_count, s.length_km, s.span_age_years,
    COALESCE(a.total_alarms, 0) AS total_alarms,
    COALESCE(a.distinct_alarm_types, 0) AS distinct_alarm_types,
    COALESCE(a.critical_alarm_count, 0) AS critical_alarm_count,
    COALESCE(a.major_alarm_count, 0) AS major_alarm_count,
    COALESCE(a.severe_alarm_count, 0) AS severe_alarm_count,
    COALESCE(a.osnr_degrade_count, 0) AS osnr_degrade_count,
    COALESCE(a.ber_high_count, 0) AS ber_high_count,
    COALESCE(a.fiber_cut_count, 0) AS fiber_cut_count,
    COALESCE(a.los_count, 0) AS los_count,
    COALESCE(a.avg_alarm_duration_hrs, 0.0) AS avg_alarm_duration_hrs,
    COALESCE(a.unack_alarm_count, 0) AS unack_alarm_count,
    COALESCE(a.days_since_last_critical, 365) AS days_since_last_critical,
    COALESCE(sr.srlg_count, 0) AS srlg_count,
    COALESCE(sr.srlg_type_count, 0) AS srlg_type_count,
    COALESCE(sr.srlg_critical_count, 0) AS srlg_critical_count,
    COALESCE(sr.srlg_severe_count, 0) AS srlg_severe_count,
    COALESCE(svc.service_count, 0) AS service_count,
    COALESCE(cond.spans_in_conduit, 1) AS spans_in_conduit
  FROM {CATALOG}.{SCHEMA}.physical_resource_span s
  LEFT JOIN (
    SELECT span_id,
      COUNT(*) AS total_alarms,
      COUNT(DISTINCT alarm_type) AS distinct_alarm_types,
      SUM(CASE WHEN severity = 'critical' THEN 1 ELSE 0 END) AS critical_alarm_count,
      SUM(CASE WHEN severity = 'major' THEN 1 ELSE 0 END) AS major_alarm_count,
      SUM(CASE WHEN severity IN ('critical','major') THEN 1 ELSE 0 END) AS severe_alarm_count,
      SUM(CASE WHEN alarm_type = 'OSNR_DEGRADE' THEN 1 ELSE 0 END) AS osnr_degrade_count,
      SUM(CASE WHEN alarm_type = 'BER_HIGH' THEN 1 ELSE 0 END) AS ber_high_count,
      SUM(CASE WHEN alarm_type = 'FIBER_CUT' THEN 1 ELSE 0 END) AS fiber_cut_count,
      SUM(CASE WHEN alarm_type = 'LOS' THEN 1 ELSE 0 END) AS los_count,
      AVG(duration_hours) AS avg_alarm_duration_hrs,
      SUM(CASE WHEN acknowledged = FALSE THEN 1 ELSE 0 END) AS unack_alarm_count,
      DATEDIFF(CURRENT_DATE(), MAX(CASE WHEN severity = 'critical' THEN alarm_time END)) AS days_since_last_critical
    FROM {CATALOG}.{SCHEMA}.resource_alarm
    GROUP BY span_id
  ) a ON s.span_id = a.span_id
  LEFT JOIN (
    SELECT span_id,
      COUNT(DISTINCT srlg_id) AS srlg_count,
      COUNT(DISTINCT srlg_type) AS srlg_type_count,
      SUM(CASE WHEN risk_severity = 'critical' THEN 1 ELSE 0 END) AS srlg_critical_count,
      SUM(CASE WHEN risk_severity IN ('critical','severe') THEN 1 ELSE 0 END) AS srlg_severe_count
    FROM {CATALOG}.{SCHEMA}.shared_risk_group
    GROUP BY span_id
  ) sr ON s.span_id = sr.span_id
  LEFT JOIN (
    SELECT span_id, COUNT(DISTINCT service_id) AS service_count
    FROM {CATALOG}.{SCHEMA}.service_path
    GROUP BY span_id
  ) svc ON s.span_id = svc.span_id
  LEFT JOIN (
    SELECT conduit_id, COUNT(*) AS spans_in_conduit
    FROM {CATALOG}.{SCHEMA}.physical_resource_span
    GROUP BY conduit_id
  ) cond ON s.conduit_id = cond.conduit_id
)
SELECT * FROM span_alarm_features
"""

baseline_df = spark.sql(baseline_query).toPandas()
print(f"Training baseline: {len(baseline_df)} spans, {len(FEATURE_COLS)} features")

# Compute baseline summary statistics
baseline_stats = {}
for col in FEATURE_COLS:
    vals = baseline_df[col].dropna()
    baseline_stats[col] = {
        "mean": vals.mean(),
        "std": vals.std(),
        "min": vals.min(),
        "max": vals.max(),
        "p25": vals.quantile(0.25),
        "p50": vals.quantile(0.50),
        "p75": vals.quantile(0.75),
        "p95": vals.quantile(0.95),
        "null_rate": baseline_df[col].isna().mean(),
        # Decile bins for PSI
        "decile_edges": list(vals.quantile(np.arange(0, 1.1, 0.1)).values)
    }

print("\nBaseline feature stats (top 5):")
for col in FEATURE_COLS[:5]:
    s = baseline_stats[col]
    print(f"  {col}: mean={s['mean']:.2f}, std={s['std']:.2f}, range=[{s['min']:.1f}, {s['max']:.1f}]")

In [0]:
def compute_psi(baseline_vals, inference_vals, bins=10):
    """Compute Population Stability Index between baseline and inference distributions.
    
    PSI < 0.1  = no significant drift
    PSI 0.1-0.25 = moderate drift (investigate)
    PSI > 0.25 = significant drift (action required)
    """
    eps = 1e-4  # avoid log(0)
    
    # Use baseline decile edges for consistent binning
    edges = np.percentile(baseline_vals, np.arange(0, 101, 100/bins))
    edges[0] = min(edges[0], np.min(inference_vals)) - 1
    edges[-1] = max(edges[-1], np.max(inference_vals)) + 1
    edges = np.unique(edges)  # Remove duplicate edges
    
    baseline_counts = np.histogram(baseline_vals, bins=edges)[0]
    inference_counts = np.histogram(inference_vals, bins=edges)[0]
    
    baseline_pct = baseline_counts / len(baseline_vals) + eps
    inference_pct = inference_counts / len(inference_vals) + eps
    
    psi = np.sum((inference_pct - baseline_pct) * np.log(inference_pct / baseline_pct))
    return psi


def compute_ks_statistic(baseline_vals, inference_vals):
    """Kolmogorov-Smirnov test statistic for distribution comparison."""
    from scipy.stats import ks_2samp
    stat, p_value = ks_2samp(baseline_vals, inference_vals)
    return stat, p_value


def assess_drift(feature_name, psi_value, ks_stat, ks_pvalue):
    """Return drift assessment for a feature."""
    if psi_value > PSI_CRITICAL:
        level = "CRITICAL"
    elif psi_value > PSI_WARNING:
        level = "WARNING"
    else:
        level = "OK"
    
    return {
        "feature": feature_name,
        "psi": round(psi_value, 4),
        "ks_stat": round(ks_stat, 4),
        "ks_pvalue": round(ks_pvalue, 4),
        "drift_level": level
    }

print("Drift detection functions defined.")

In [0]:
import json
from datetime import datetime, timedelta

# ── Read inference table (AI Gateway format) ──
try:
    inference_raw = spark.sql(f"""
        SELECT * FROM {INFERENCE_TABLE}
        ORDER BY timestamp_ms DESC
    """)
    inference_count = inference_raw.count()
    print(f"Inference table rows: {inference_count}")
    
    if inference_count > 0:
        # Show schema
        inference_raw.printSchema()
        inference_raw.show(3, truncate=80)
        
        # Parse features from request payload
        # AI Gateway inference tables store requests as JSON in the 'request' column
        inference_pd = inference_raw.toPandas()
        
        # Extract features from the request column
        parsed_features = []
        for _, row in inference_pd.iterrows():
            try:
                req = json.loads(row['request']) if isinstance(row['request'], str) else row['request']
                if 'dataframe_split' in req:
                    cols = req['dataframe_split']['columns']
                    data_rows = req['dataframe_split']['data']
                    for d in data_rows:
                        parsed_features.append(dict(zip(cols, d)))
                elif 'dataframe_records' in req:
                    parsed_features.extend(req['dataframe_records'])
            except Exception:
                continue
        
        if parsed_features:
            inference_features = pd.DataFrame(parsed_features)
            print(f"\nParsed {len(inference_features)} inference feature vectors")
            print(f"Columns: {list(inference_features.columns)}")
        else:
            print("\nNo parseable feature vectors found in requests")
            inference_features = pd.DataFrame()
    else:
        print("Inference table empty — requests may still be propagating (2-5 min delay)")
        inference_features = pd.DataFrame()
        
except Exception as e:
    print(f"Inference table not yet created or accessible: {e}")
    print("This is expected — the table is created after the first request is logged (2-5 min delay).")
    inference_features = pd.DataFrame()

In [0]:
# ── Feature Drift Report ──
alerts = []

if len(inference_features) > 0:
    drift_results = []
    
    for col in FEATURE_COLS:
        if col in inference_features.columns:
            inf_vals = inference_features[col].dropna().astype(float).values
            base_vals = baseline_df[col].dropna().astype(float).values
            
            if len(inf_vals) >= 5:  # Need minimum samples
                psi = compute_psi(base_vals, inf_vals)
                ks_stat, ks_pval = compute_ks_statistic(base_vals, inf_vals)
                result = assess_drift(col, psi, ks_stat, ks_pval)
                drift_results.append(result)
                
                if result['drift_level'] in ('WARNING', 'CRITICAL'):
                    alerts.append(f"DRIFT-{result['drift_level']}: {col} PSI={result['psi']}")
            else:
                drift_results.append({
                    "feature": col, "psi": None, "ks_stat": None,
                    "ks_pvalue": None, "drift_level": "INSUFFICIENT_DATA"
                })
    
    drift_df = pd.DataFrame(drift_results)
    
    # Summary
    print("="*70)
    print("FEATURE DRIFT REPORT — Lakelink Fiber Span Risk Classifier v2")
    print("="*70)
    print(f"Baseline: {len(baseline_df)} training spans")
    print(f"Inference: {len(inference_features)} scored requests")
    print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print()
    
    for _, row in drift_df.iterrows():
        status_icon = {"OK": "✅", "WARNING": "⚠️", "CRITICAL": "🚨", "INSUFFICIENT_DATA": "❓"}
        icon = status_icon.get(row['drift_level'], '?')
        psi_str = f"{row['psi']:.4f}" if row['psi'] is not None else "N/A"
        print(f"  {icon} {row['feature']:30s} PSI={psi_str:>8s}  {row['drift_level']}")
    
    # Counts
    print(f"\nSummary: {len(drift_df[drift_df.drift_level=='OK'])} OK, "
          f"{len(drift_df[drift_df.drift_level=='WARNING'])} WARNING, "
          f"{len(drift_df[drift_df.drift_level=='CRITICAL'])} CRITICAL")
else:
    print("⚠️ Inference data not yet available — feature drift analysis deferred.")
    print("Re-run this notebook after inference table populates (2-5 min after first request).")

In [0]:
# ── Prediction Distribution Monitor ──
if len(inference_features) > 0:
    try:
        # Parse predictions from response column
        predictions = []
        for _, row in inference_pd.iterrows():
            try:
                resp = json.loads(row['response']) if isinstance(row['response'], str) else row['response']
                if 'predictions' in resp:
                    predictions.extend(resp['predictions'])
            except Exception:
                continue
        
        if predictions:
            preds = np.array(predictions)
            risk_rate = np.mean(preds == 1)
            
            print("="*70)
            print("PREDICTION DISTRIBUTION REPORT")
            print("="*70)
            print(f"Total predictions: {len(preds)}")
            print(f"Risk rate: {risk_rate:.1%} (expected range: {RISK_RATE_LOW:.0%}-{RISK_RATE_HIGH:.0%})")
            print(f"  Class 0 (normal): {np.sum(preds == 0)} ({np.mean(preds == 0):.1%})")
            print(f"  Class 1 (risk):   {np.sum(preds == 1)} ({np.mean(preds == 1):.1%})")
            
            # Training baseline risk rate
            # Training had ~5.65% positive rate (from run params)
            training_risk_rate = 0.0565
            print(f"\nTraining baseline risk rate: {training_risk_rate:.1%}")
            print(f"Current inference risk rate: {risk_rate:.1%}")
            print(f"Deviation: {abs(risk_rate - training_risk_rate):.1%}")
            
            if risk_rate < RISK_RATE_LOW:
                alerts.append(f"PRED-LOW: Risk rate {risk_rate:.1%} below threshold {RISK_RATE_LOW:.0%}")
                print(f"\n🚨 ALERT: Risk rate unusually LOW — model may be undertriggering")
            elif risk_rate > RISK_RATE_HIGH:
                alerts.append(f"PRED-HIGH: Risk rate {risk_rate:.1%} above threshold {RISK_RATE_HIGH:.0%}")
                print(f"\n🚨 ALERT: Risk rate unusually HIGH — possible data shift or model degradation")
            else:
                print(f"\n✅ Risk rate within expected bounds")
        else:
            print("No predictions found in response data")
    except Exception as e:
        print(f"Prediction parsing error: {e}")
else:
    print("⚠️ Prediction distribution analysis deferred — waiting for inference data.")

In [0]:
# ── Latency SLA Monitor ──
if inference_count > 0:
    try:
        latency_df = inference_raw.select(
            F.col("timestamp_ms"),
            (F.col("execution_time_ms")).alias("latency_ms")
        ).filter(F.col("execution_time_ms").isNotNull()).toPandas()
        
        if len(latency_df) > 0:
            print("="*70)
            print("LATENCY SLA REPORT")
            print("="*70)
            p50 = latency_df['latency_ms'].quantile(0.50)
            p95 = latency_df['latency_ms'].quantile(0.95)
            p99 = latency_df['latency_ms'].quantile(0.99)
            avg = latency_df['latency_ms'].mean()
            
            print(f"Requests measured: {len(latency_df)}")
            print(f"Avg latency:  {avg:.1f} ms")
            print(f"P50 latency:  {p50:.1f} ms")
            print(f"P95 latency:  {p95:.1f} ms  (SLA: {LATENCY_P95_SLA_MS} ms)")
            print(f"P99 latency:  {p99:.1f} ms")
            
            if p95 > LATENCY_P95_SLA_MS:
                alerts.append(f"LATENCY-SLA: P95={p95:.0f}ms exceeds {LATENCY_P95_SLA_MS}ms SLA")
                print(f"\n🚨 ALERT: P95 latency exceeds SLA threshold!")
            else:
                print(f"\n✅ Latency within SLA")
        else:
            print("No latency data available in inference table")
    except Exception as e:
        print(f"Latency column not found or error: {e}")
        print("Note: Column names may differ in AI Gateway inference tables.")
else:
    print("⚠️ Latency analysis deferred — waiting for inference data.")

In [0]:
# ── Data Quality Checks ──
if len(inference_features) > 0:
    print("="*70)
    print("DATA QUALITY REPORT")
    print("="*70)
    
    dq_issues = []
    
    for col in FEATURE_COLS:
        if col in inference_features.columns:
            vals = inference_features[col]
            null_rate = vals.isna().mean()
            
            # Null check
            if null_rate > NULL_THRESHOLD:
                dq_issues.append(f"NULL: {col} has {null_rate:.1%} nulls (threshold: {NULL_THRESHOLD:.0%})")
                alerts.append(f"DQ-NULL: {col} null rate {null_rate:.1%}")
            
            # Range check against baseline
            if col in baseline_stats:
                bs = baseline_stats[col]
                non_null = vals.dropna().astype(float)
                if len(non_null) > 0:
                    below_min = (non_null < bs['min']).sum()
                    above_max = (non_null > bs['max']).sum()
                    if below_min > 0 or above_max > 0:
                        dq_issues.append(f"RANGE: {col} has {below_min} below min, {above_max} above max vs training")
    
    if dq_issues:
        for issue in dq_issues:
            print(f"  ⚠️ {issue}")
    else:
        print("  ✅ All features pass null and range checks")
    
    # Schema check — are all expected features present?
    missing_features = [c for c in FEATURE_COLS if c not in inference_features.columns]
    if missing_features:
        print(f"\n  🚨 MISSING FEATURES: {missing_features}")
        alerts.append(f"DQ-SCHEMA: Missing features: {missing_features}")
    else:
        print(f"\n  ✅ All {len(FEATURE_COLS)} expected features present")
else:
    print("⚠️ Data quality checks deferred — waiting for inference data.")

In [0]:
# ── Monitoring Summary ──
from datetime import datetime

print("="*70)
print("MONITORING SUMMARY — Lakelink Fiber Span Risk Classifier v2")
print(f"Run: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)

if alerts:
    print(f"\n🚨 {len(alerts)} ALERT(S) RAISED:")
    for alert in alerts:
        print(f"  • {alert}")
    
    # Store alerts in a Delta table for historical tracking
    alert_rows = [{"timestamp": datetime.now().isoformat(), "alert": a, "endpoint": ENDPOINT_NAME} for a in alerts]
    alert_df = spark.createDataFrame(alert_rows)
    alert_df.write.mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.model_monitoring_alerts")
    print(f"\n💾 Alerts persisted to {CATALOG}.{SCHEMA}.model_monitoring_alerts")
else:
    print("\n✅ ALL CLEAR — No alerts triggered")
    print("  • Feature drift: within bounds")
    print("  • Prediction distribution: stable")
    print("  • Latency: within SLA")
    print("  • Data quality: clean")

print(f"\nEndpoint: {ENDPOINT_NAME}")
print(f"Model: {MODEL_NAME} v{MODEL_VERSION}")
print(f"Training baseline: {len(baseline_df)} spans")
if len(inference_features) > 0:
    print(f"Inference samples: {len(inference_features)}")
print("\nNext run: scheduled or on-demand")